In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from pprint import pprint

import src.utils as utils
import src.visuals as visual

from src.models import PINN
from src.loss import NavierStokesLoss
from src.dataloader import load_data, gen_dataloaders
from src.train import train_collocation

In [ ]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

TRAIN_DATA_PATH = pathlib.Path("data/real_data/frac_1")
TRAIN_FILE_NAME = "data"

TEST_DATA_PATH = pathlib.Path("data/raw_data")
TEST_FILE_NAME = "full_data_v2"

INPUT_COL_NAMES = ["time", "re", "x", "y"]
TARGET_COL_NAMES = ["U_x", "U_y", "p"]

BOX = {
    "x_min": 0.25,
    "x_max": 0.75,
    "y_min": 0.25,
    "y_max": 0.75,
}

EPOCHS = 300
BATCH_SIZE = 256
LEARNING_RATE = 1e-3

N_COLLOCATION = 4096

PHYSICS_WEIGHTS = [
    0.0,
    0.01,
    0.05,
    0.1,
    0.5,
    1.0
]

LR_FACTOR = 0.5
LR_PATIENCE = 15

print(f"Datasets: {TRAIN_FILE_NAME}, {TEST_FILE_NAME}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Collocation points: {N_COLLOCATION}")
print(f"Physics weights: {PHYSICS_WEIGHTS}")
print(f"Held-out box: {BOX}")

In [ ]:
train_df, valid_df, _ = load_data(TRAIN_DATA_PATH, TRAIN_FILE_NAME)
_, _, test_df = load_data(TEST_DATA_PATH, TEST_FILE_NAME)

print(f"Train samples:      {len(train_df):,}")
print(f"Validation samples: {len(valid_df):,}")
print(f"Test samples:       {len(test_df):,}")

train_re = list(map(int, sorted(train_df["re"].unique())))
valid_re = list(map(int, sorted(valid_df["re"].unique())))
test_re = list(map(int, sorted(test_df["re"].unique())))

time_steps = list(map(float, sorted(train_df["time"].unique())))

print(f"\nTraining Reynolds numbers:   {train_re}")
print(f"Validation Reynolds numbers: {valid_re}")
print(f"Test Reynolds numbers:       {test_re}")

print(f"\nTime steps: {time_steps}")

In [ ]:
_, train_out = utils.split_by_box(train_df, BOX)

test_inside, test_outside = utils.split_by_box(test_df, BOX)

train_re_values = sorted(train_out["re"].unique())
valid_re_values = sorted(valid_df["re"].unique())

print(f"Original train samples: {len(train_df):,}")
print(f"Train after held-out:   {len(train_out):,}")
print(f"Removed from train:     {len(train_df) - len(train_out):,}")

print(f"\nValidation samples:     {len(valid_df):,}")

print(f"\nTest samples:")
print(f"  Inside box:           {len(test_inside):,}")
print(f"  Outside box:          {len(test_outside):,}")

removed_pct = 100 * (len(train_df) - len(train_out)) / len(train_df)

print(f"\nRemoved from training:  {removed_pct:.2f}%")

In [ ]:
mean = train_out.mean()
std = train_out.std()

train_df_norm = utils.normalize_data(train_out, mean, std)
valid_df_norm = utils.normalize_data(valid_df, mean, std)

In [ ]:
train_dataloader, valid_dataloader, _ = gen_dataloaders(
    train_df_norm,
    valid_df_norm,
    valid_df_norm,
    INPUT_COL_NAMES,
    TARGET_COL_NAMES,
    BATCH_SIZE,
)

In [ ]:
ranges = utils.get_domain_ranges(
    train_df,
    INPUT_COL_NAMES,
    overrides={
        "x": (0.0, 1.0),
        "y": (0.0, 1.0),
    },
)

In [ ]:
results = []

for c_physics in PHYSICS_WEIGHTS:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    print("\n" + "=" * 80)
    print(f"Training model with c_physics = {c_physics}")
    print("=" * 80)

    tag = f"cphys_{c_physics:g}"

    run_dir = utils.create_run_directory(label=f"heldout_{tag}")

    config = {
        "experiment": "heldout_region",
        "physics_weight": c_physics,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "collocation_points": N_COLLOCATION,
        "seed": SEED,
        "box": BOX,
        "train_samples": len(train_out),
        "valid_samples": len(valid_df),
        "test_inside_samples": len(test_inside),
        "test_outside_samples": len(test_outside),
    }

    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=4)

    model = PINN(
        len(INPUT_COL_NAMES),
        len(TARGET_COL_NAMES),
    ).to(device)

    criterion = NavierStokesLoss(
        c_physics,
        mean,
        std,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
    )

    history = train_collocation(
        model=model,
        train_dataloader=train_dataloader,
        valid_dataloader=valid_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=EPOCHS,
        run_dir=run_dir,
        input_col_names=INPUT_COL_NAMES,
        ranges=ranges,
        mean=mean,
        std=std,
        n_collocation=N_COLLOCATION,
        train_re_values=train_re_values,
        valid_re_values=valid_re_values,
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)

    fig, _ = utils.plot_training_history(
        history_df,
        output_path=run_dir / "losses.png",
        title=f"Held-out (c_physics={c_physics})",
    )
    plt.close(fig)

    checkpoint = torch.load(
        run_dir / "best_model.pth",
        map_location=device,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    metrics = []

    for region_name, region_df in [
        ("inside_box", test_inside),
        ("outside_box", test_outside),
    ]:

        region_metrics = utils.evaluate_model(
            model=model,
            df=region_df,
            input_col_names=INPUT_COL_NAMES,
            target_col_names=TARGET_COL_NAMES,
            mean=mean,
            std=std,
            device=device,
        )

        row = {
            "tag": tag,
            "c_physics": c_physics,
            "region": region_name,
            "n_points": len(region_df),
        }

        row.update(utils.flatten_metrics(region_metrics))

        metrics.append(row)
        results.append(row)

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(run_dir / "metrics.csv", index=False)

    print(metrics_df[["region", "R2_all", "MAE_all", "RMSE_all"]])

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["c_physics", "region"]
).reset_index(drop=True)

results_df.to_csv("heldout_results.csv", index=False)

display(results_df)

### TODO: Vizuelizacija